In [2]:
import requests
import json
from typing import List, Tuple

class OllamaCrossEncoder:
    """
    A CrossEncoder-like class using Ollama for pairwise scoring.
    This can be used for reranking search results or classification tasks.
    """

    def __init__(self, model: str = "llama3.2"):
        self.model = model
        self.api_url = "http://localhost:11434/api/generate"  # Ollama local API endpoint

    def score_pairs(self, pairs: List[Tuple[str, str]]) -> List[float]:
        """
        Scores text pairs using Ollama.
        Returns a list of similarity/confidence scores between 0 and 1.
        """
        scores = []
        for query, candidate in pairs:
            prompt = (
                f"Rate the relevance between the following two texts on a scale from 0 to 1:\n"
                f"Query: {query}\n"
                f"Candidate: {candidate}\n"
                f"Answer with only the numeric score."
            )

            try:
                response = requests.post(
                    self.api_url,
                    json={"model": self.model, "prompt": prompt, "stream": False},
                    timeout=30
                )
                response.raise_for_status()
                result_text = response.json().get("response", "").strip()

                # Parse score safely
                try:
                    score = float(result_text)
                    if 0 <= score <= 1:
                        scores.append(score)
                    else:
                        scores.append(0.0)  # fallback if out of range
                except ValueError:
                    scores.append(0.0)  # fallback if parsing fails

            except requests.RequestException as e:
                print(f"Error contacting Ollama: {e}")
                scores.append(0.0)

        return scores

    def rerank(self, query: str, candidates: List[str]) -> List[Tuple[str, float]]:
        """
        Reranks candidates based on relevance to the query.
        Returns a list of (candidate, score) sorted by score descending.
        """
        pairs = [(query, c) for c in candidates]
        scores = self.score_pairs(pairs)
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return ranked

In [17]:
import os

from typing import TypedDict, List

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma

from langgraph.graph import StateGraph, START, END

# =========================================================
# 2. KNOWLEDGE BASE
# =========================================================

documents = [

    Document(
        page_content="""
        LangGraph is a framework for building stateful applications
        with large language models. Applications are represented as
        graphs containing nodes and edges.
        """,
        metadata={"topic": "LangGraph"}
    ),

    Document(
        page_content="""
        LangGraph allows applications to maintain state between
        different nodes. Nodes can read the current state and
        update it during graph execution.
        """,
        metadata={"topic": "State"}
    ),

    Document(
        page_content="""
        LangGraph persistence allows applications to save graph state.
        Checkpointers store snapshots of the state during execution.
        This allows a graph to resume from a previous state.
        """,
        metadata={"topic": "Persistence"}
    ),

    Document(
        page_content="""
        LangGraph can maintain conversational memory by using
        persistence and checkpoints. Previous conversation state
        can be retrieved when the user returns to a conversation.
        """,
        metadata={"topic": "Memory"}
    ),

    Document(
        page_content="""
        LangGraph supports human-in-the-loop workflows. A graph can
        pause execution and wait for human approval before continuing.
        """,
        metadata={"topic": "Human-in-the-loop"}
    ),

    Document(
        page_content="""
        LangGraph can implement agentic RAG systems using nodes for
        retrieval, generation, tool execution and conditional routing.
        """,
        metadata={"topic": "RAG"}
    )
]


# =========================================================
# 3. VECTOR DATABASE
# =========================================================

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)


# =========================================================
# 4. LLM
# =========================================================

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)



# =========================================================
# 6. ANSWER PROMPT
# =========================================================

answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful RAG assistant.

Answer the user's question using ONLY the provided context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say that there is not enough information.

Answer:
""")


answer_chain = answer_prompt | llm


# =========================================================
# 7. LANGGRAPH STATE
# =========================================================

class RAGState(TypedDict):
    question: str
    documents: List[Document]


# =========================================================
# 9. RETRIEVAL NODE
# =========================================================

def retrieve_documents(state: RAGState):

    return {
        "documents": retriever.invoke(state["question"])
    }


# =========================================================
# 10. DEDUPLICATION NODE
# =========================================================

def deduplicate_documents(state: RAGState):

    unique_documents = {}

    for doc in state["documents"]:

        key = doc.page_content

        unique_documents[key] = doc

    return {
        "documents": list(unique_documents.values())
    }


# =========================================================
# 11. ANSWER GENERATION NODE
# =========================================================

def rerank_documents(state: RAGState):

    encoder = OllamaCrossEncoder(model="llama3.2")

    query = state["question"]
    candidates = state["documents"]

    ranked_results = encoder.rerank(query, candidates)

    print("\nReranked Results:")
    for text, score in ranked_results:
        print(f"Score: {score:.2f} - {text}")
        


# =========================================================
# 12. BUILD LANGGRAPH
# =========================================================

builder = StateGraph(RAGState)


builder.add_node(
    "rerank_documents",
    rerank_documents
)

builder.add_node(
    "retrieve_documents",
    retrieve_documents
)

builder.add_node(
    "deduplicate_documents",
    deduplicate_documents
)



builder.add_edge(
    START,
    "retrieve_documents"
)

builder.add_edge(
    "retrieve_documents",
    "deduplicate_documents"
)

builder.add_edge(
    "deduplicate_documents",
    "rerank_documents"
)

builder.add_edge(
    "rerank_documents",
    END
)


graph = builder.compile()


# =========================================================
# 13. EXECUTE
# =========================================================

result = graph.invoke({
    "question": "How does LangGraph handle memory?"
})


# =========================================================
# 14. DISPLAY RESULTS
# =========================================================

print("\n===== ORIGINAL QUESTION =====")

print(result["question"])







Reranked Results:
Score: 0.50 - page_content='
        LangGraph is a framework for building stateful applications
        with large language models. Applications are represented as
        graphs containing nodes and edges.
        ' metadata={'topic': 'LangGraph'}

===== ORIGINAL QUESTION =====
How does LangGraph handle memory?
